# 6.23 — Stochastic Depth & Spectral Normalization

Stochastic depth and spectral normalization are two ways to keep very deep networks useful instead of merely large. Stochastic depth randomly drops whole residual branches during training so paths do not co-adapt, while spectral normalization rescales a weight matrix by its largest singular value so one layer cannot amplify signals and gradients without bound.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build stochastic depth and spectral normalization one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is shown with tiny NumPy arrays so the path survival variable and spectral rescale are not black boxes. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random masks, and linear algebra for singular values.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for every sampled survival mask.

### 1. Residual paths: the unit that stochastic depth gates

A residual block adds a learned branch `F(x)` back to the shortcut `x`. Without the shortcut, every layer must carry the whole signal; with the shortcut, the branch only has to learn a correction. Stochastic depth acts on exactly this correction path: sometimes use `y = x + F(x)`, sometimes use `y = x`, so first we inspect the ungated residual arithmetic.

In [ ]:
x_w = np.array([1.5, -0.5])  # two-input activation vector from the lesson text.
W_res_w = np.array([[1.6, 0.2], [-0.4, 1.1]])  # tiny residual-branch weight matrix.
b_res_w = np.array([0.8, -0.2])  # residual-branch bias.
pre_w = W_res_w @ x_w + b_res_w  # affine signal before the nonlinearity.
F_w = np.maximum(pre_w, 0.0)  # ReLU branch output.
print("affine signal:", np.round(pre_w, 3))
print("residual branch F(x):", np.round(F_w, 3))
assert round(float(pre_w[0]), 3) == 3.100

▶ What you'll see: the first branch coordinate is exactly `3.100`, matching the lesson's scratch pass.

In [ ]:
y_plain_w = x_w + F_w  # ordinary residual output when the branch is active.
print("shortcut x:", x_w)
print("x + F(x):", np.round(y_plain_w, 3))
plt.figure(figsize=(4.6, 3))
plt.bar(["x0", "x1", "F0", "F1"], [x_w[0], x_w[1], F_w[0], F_w[1]], color=["gray", "gray", "teal", "teal"])
plt.axhline(0, color="black", linewidth=0.7)
plt.title("1: shortcut plus residual correction")
plt.ylabel("activation value")
plt.show()

▶ What you'll see: the residual branch contributes a positive correction while the shortcut preserves the original signal.

*Why it's done this way:* A residual block is easy to gate because the shortcut already defines a safe fallback. If a plain layer disappears, the network loses the whole transformation; if a residual branch disappears, the block becomes the identity map `y=x`, so signal and gradient still have a direct route through depth.

### 2. Stochastic depth: sample a path, then rescale it

Stochastic depth samples a Bernoulli survival variable `b`: `b=1` keeps the residual branch and `b=0` drops it. During training we usually use inverted scaling, `y = x + (b/p)F(x)`, where `p` is the survival probability. The division by `p` keeps the expected branch contribution equal to `F(x)`, because `E[b/p] = p/p = 1`.

In [ ]:
p_survive_w = 0.8  # survival probability from the lesson's path-survival idea.
mask_values_w = np.array([0.0, 1.0])  # dropped path and surviving path.
outcomes_w = np.array([x_w + (m / p_survive_w) * F_w for m in mask_values_w])  # two possible outputs.
print("drop output:", np.round(outcomes_w[0], 3))
print("keep output with 1/p scaling:", np.round(outcomes_w[1], 3))

▶ What you'll see: dropping gives the shortcut `x`; keeping gives a larger-than-usual branch because it is scaled by `1/0.8`.

In [ ]:
expected_w = (1 - p_survive_w) * outcomes_w[0] + p_survive_w * outcomes_w[1]  # expectation over the Bernoulli mask.
plain_expected_w = x_w + F_w  # output of the ordinary residual block.
print("expected stochastic-depth output:", np.round(expected_w, 3))
print("plain residual output:", np.round(plain_expected_w, 3))
assert np.allclose(expected_w, plain_expected_w)

▶ What you'll see: the average stochastic-depth output exactly matches the ungated residual output.

In [ ]:
rng_w = np.random.default_rng(0)
samples_w = rng_w.random(2000) < p_survive_w  # many Bernoulli survival draws.
branch_scale_w = samples_w.astype(float) / p_survive_w  # 0 or 1/p.
mean_scale_w = float(branch_scale_w.mean())
print("empirical survival rate:", round(float(samples_w.mean()), 3))
print("empirical mean branch scale:", round(mean_scale_w, 3))
plt.figure(figsize=(4.6, 3))
plt.hist(branch_scale_w, bins=[-0.05, 0.05, 1.20, 1.30], color="mediumpurple", edgecolor="black")
plt.title("2: stochastic-depth branch scale")
plt.xlabel("multiplier on F(x)")
plt.ylabel("count")
plt.show()

▶ What you'll see: most samples keep the branch at scale `1.25`, some drop it to `0`, and the mean scale is near `1`.

*Why it's done this way:* The mask creates random shorter networks, which regularizes path co-adaptation. The `1/p` factor is not cosmetic: without it, the train-time average would be `x+pF(x)` while test time would use `x+F(x)`, creating a scale mismatch exactly where deep networks are already fragile.

### 3. Depth schedules: shallow blocks survive more often than deep blocks

In practice, survival probability is often high near the input and lower near the output. Early blocks process low-level features that many later computations depend on, so dropping them aggressively can scramble the representation. Later residual branches can be dropped more often, producing many effective depths while keeping the input pipeline stable.

In [ ]:
L_w = 8  # number of residual blocks in a tiny network.
p_start_w, p_end_w = 1.0, 0.6  # linear survival schedule from shallow to deep.
p_layers_w = np.linspace(p_start_w, p_end_w, L_w)  # per-layer survival probabilities.
print("survival probabilities:", np.round(p_layers_w, 3))
assert round(float(p_layers_w[-1]), 3) == 0.600

▶ What you'll see: the first block always survives and the deepest block survives 60% of the time.

In [ ]:
rng_sched_w = np.random.default_rng(1)
trials_w = 5000
masks_w = rng_sched_w.random((trials_w, L_w)) < p_layers_w  # sample many training networks.
effective_depth_w = masks_w.sum(axis=1)  # number of active residual branches per pass.
print("mean active branches:", round(float(effective_depth_w.mean()), 3))
print("expected active branches:", round(float(p_layers_w.sum()), 3))
assert abs(float(effective_depth_w.mean()) - float(p_layers_w.sum())) < 0.08

▶ What you'll see: empirical active depth matches the sum of survival probabilities.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(1, L_w + 1), p_layers_w, color="seagreen")
plt.ylim(0, 1.05)
plt.title("3: survival schedule by depth")
plt.xlabel("residual block")
plt.ylabel("survival probability")
plt.show()

▶ What you'll see: a descending survival schedule — shallow paths are protected, deeper paths are regularized harder.

*Why it's done this way:* The expected active depth is `sum_l p_l`, so the schedule is a direct capacity knob. Lowering deep survival probabilities trains an ensemble of shorter paths, while keeping early probabilities high prevents every minibatch from seeing a completely different input representation.

### 4. Spectral norm: the largest stretch of a layer

A weight matrix can stretch some input directions more than others. The largest possible stretch is the largest singular value, `σ_max(W)`. Spectral normalization divides by that value, `W_bar = W / σ_max(W)`, so the normalized layer has maximum gain 1. That is a Lipschitz control: distances and gradients cannot be amplified by this linear layer more than the chosen bound.

In [ ]:
W_w = np.array([[3.0, 1.0], [0.0, 2.0]])  # a small layer with visible anisotropic stretch.
U_w, s_w, Vt_w = np.linalg.svd(W_w, full_matrices=False)  # exact singular values for the tiny demo.
sigma_w = float(s_w[0])  # largest singular value.
W_bar_w = W_w / sigma_w  # spectral normalization to unit top singular value.
print("singular values:", np.round(s_w, 3))
print("sigma_max:", round(sigma_w, 3))
assert round(sigma_w, 3) == 3.257

▶ What you'll see: the top singular value is about `3.257`, so the raw layer can stretch some direction by more than 3×.

In [ ]:
s_bar_w = np.linalg.svd(W_bar_w, compute_uv=False)  # singular values after rescaling.
print("normalized singular values:", np.round(s_bar_w, 3))
assert round(float(s_bar_w[0]), 3) == 1.000
angles_w = np.linspace(0, 2 * np.pi, 160)
circle_w = np.vstack([np.cos(angles_w), np.sin(angles_w)])
raw_img_w = W_w @ circle_w
norm_img_w = W_bar_w @ circle_w
plt.figure(figsize=(4.8, 4))
plt.plot(circle_w[0], circle_w[1], color="gray", label="unit circle")
plt.plot(raw_img_w[0], raw_img_w[1], color="crimson", label="W circle")
plt.plot(norm_img_w[0], norm_img_w[1], color="teal", label="W_bar circle")
plt.axis("equal"); plt.legend(); plt.title("4: spectral norm caps stretch")
plt.show()

▶ What you'll see: the raw matrix turns the unit circle into a large ellipse; the normalized matrix's largest radius is capped near 1.

*Why it's done this way:* Singular values measure stretch along orthogonal directions. Dividing by only `σ_max` preserves the matrix's directional pattern but rescales its worst-case gain, so the layer still learns a transformation while respecting a hard scale budget.

### 5. Power iteration: estimating σmax without a full SVD

Real neural layers are too large to run a full SVD at every update. Power iteration estimates the top singular vectors by repeatedly multiplying by `W` and `W.T`. Each iteration aligns a vector more with the direction of maximum stretch, then the scalar `u^T W v` estimates `σ_max`.

In [ ]:
u_w = np.array([1.0, 0.0])  # initial left vector for power iteration.
trace_sigma_w = []  # store estimates so convergence is visible.
for step_w in range(6):
    v_w = W_w.T @ u_w
    v_w = v_w / np.linalg.norm(v_w)
    u_w = W_w @ v_w
    u_w = u_w / np.linalg.norm(u_w)
    trace_sigma_w.append(float(u_w @ W_w @ v_w))
print("power-iteration estimates:", np.round(trace_sigma_w, 4))
print("exact sigma_max:", round(sigma_w, 4))
assert abs(trace_sigma_w[-1] - sigma_w) < 0.01

▶ What you'll see: the estimate quickly approaches the exact top singular value.

In [ ]:
W_pi_w = W_w / trace_sigma_w[-1]  # normalize with the estimated spectral norm.
gain_pi_w = np.linalg.svd(W_pi_w, compute_uv=False)[0]
print("estimated-normalized top gain:", round(float(gain_pi_w), 3))
plt.figure(figsize=(4.6, 3))
plt.plot(trace_sigma_w, marker="o", color="darkorange")
plt.axhline(sigma_w, color="black", linestyle="--", label="exact")
plt.title("5: power iteration finds σmax")
plt.xlabel("iteration")
plt.ylabel("σ estimate")
plt.legend()
plt.show()

▶ What you'll see: the estimate rises to the dashed exact value, so the estimated normalization has top gain near 1.

*Why it's done this way:* Power iteration is cheap because it uses matrix-vector products instead of decomposing the whole matrix. The repeated normalize-multiply steps amplify the dominant singular direction and suppress weaker directions, making the largest stretch visible with very little bookkeeping.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, Bernoulli masks, singular values, and small numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for every heatmap, bar chart, curve, and geometric diagnostic.
np.random.seed(0) # make random examples reproducible across notebook runs.

def relu(z): # define the ReLU nonlinearity used by tiny residual branches.
    return np.maximum(z, 0.0) # keep positive coordinates and zero out negative coordinates.

def residual_branch(x, W, b): # define a tiny branch F(x) = ReLU(Wx+b).
    return relu(W @ x + b) # return the branch correction vector.

def stochastic_depth_output(x, F, survive, p): # define inverted stochastic-depth residual output.
    return x + (survive / p) * F # keep expectation equal to x+F when survive is Bernoulli(p).

def spectral_normalize(W): # define exact spectral normalization for small teaching matrices.
    sigma = np.linalg.svd(W, compute_uv=False)[0] # compute the largest singular value.
    return W / sigma, sigma # return the rescaled matrix and its original top gain.

def power_sigma(W, steps=8): # estimate the largest singular value with power iteration.
    u = np.ones(W.shape[0]) # initialize a deterministic nonzero left vector.
    u = u / np.linalg.norm(u) # normalize so the scale does not grow across iterations.
    estimates = [] # store intermediate sigma estimates for inspection.
    for _ in range(steps): # repeat alternating W^T and W multiplications.
        v = W.T @ u # move to a right-vector estimate.
        v = v / np.linalg.norm(v) # normalize the right vector.
        u = W @ v # move back to a left-vector estimate.
        u = u / np.linalg.norm(u) # normalize the left vector.
        estimates.append(float(u @ W @ v)) # Rayleigh-style singular value estimate.
    return estimates[-1], np.array(estimates) # return the final estimate and convergence trace.

## 🟢 Basics (warm-up)

### Basic 1 — Build a residual branch

**Goal.** Compute a tiny branch `F(x)=ReLU(Wx+b)`, because stochastic depth gates residual corrections rather than the shortcut itself. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.5, -0.5]) # define the two-coordinate activation vector.
W_b1 = np.array([[1.6, 0.2], [-0.4, 1.1]]) # define a small residual-branch weight matrix.
b_b1 = np.array([0.8, -0.2]) # define a bias so the first affine coordinate matches the lesson arithmetic.
pre_b1 = W_b1 @ x_b1 + b_b1 # compute Wx+b before the gate.
print("pre-activation:", np.round(pre_b1, 3)) # inspect the affine signal.
assert round(float(pre_b1[0]), 3) == 3.100 # verify 1.6*1.5 + 0.2*(-0.5) + 0.8.

▶ What you'll see: the first pre-activation is `3.100`, while the second is negative.

In [ ]:
F_b1 = relu(pre_b1) # apply ReLU to get the residual correction.
y_b1 = x_b1 + F_b1 # add the correction to the shortcut.
print("F(x):", np.round(F_b1, 3)) # inspect the branch after ReLU.
print("x + F(x):", np.round(y_b1, 3)) # inspect the full residual output.
plt.figure(figsize=(4, 3))
plt.bar(["x0", "x1", "F0", "F1"], [x_b1[0], x_b1[1], F_b1[0], F_b1[1]], color=["gray", "gray", "teal", "teal"])
plt.axhline(0, color="black", linewidth=0.7)
plt.title("Basic 1: residual pieces")
plt.show()

▶ What you'll see: ReLU removes the negative branch coordinate, leaving only one correction.

👀 Takeaway: stochastic depth works cleanly because a residual block can always fall back to the shortcut.

### Basic 2 — Drop or keep one branch

**Goal.** Apply the Bernoulli survival variable to one branch, because the stochastic-depth formula is `y=x+bF(x)` before any expectation correction. We build it in 2 steps.

In [ ]:
x_b2 = np.array([1.5, -0.5]) # define the shortcut signal.
F_b2 = np.array([3.1, 0.0]) # reuse the worked residual correction.
b_drop_b2 = 0.0 # branch is dropped.
b_keep_b2 = 1.0 # branch survives.
print("drop output:", x_b2 + b_drop_b2 * F_b2) # inspect y=x.
print("keep output:", x_b2 + b_keep_b2 * F_b2) # inspect y=x+F.

▶ What you'll see: the dropped block returns the shortcut, and the kept block returns the ordinary residual output.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["drop", "keep"], [(x_b2 + b_drop_b2 * F_b2)[0], (x_b2 + b_keep_b2 * F_b2)[0]], color=["gray", "seagreen"])
plt.title("Basic 2: first coordinate with b=0 or b=1")
plt.ylabel("output coordinate 0")
plt.show()

▶ What you'll see: keeping the branch lifts coordinate 0 by exactly the branch value.

👀 Takeaway: the survival mask changes network depth by removing residual corrections, not by deleting the identity path.

### Basic 3 — Preserve expectation with inverted scaling

**Goal.** Verify `E[(b/p)F]=F`, because train-time random dropping should match test-time residual scale on average. We build it in 3 steps.

In [ ]:
p_b3 = 0.8 # branch survival probability.
F_b3 = np.array([3.1, 0.0]) # branch correction.
scale_keep_b3 = 1.0 / p_b3 # inverted stochastic-depth scale when branch survives.
print("keep scale:", scale_keep_b3) # inspect 1/p.
assert round(scale_keep_b3, 3) == 1.250 # verify the numeric scale.

▶ What you'll see: a surviving branch is multiplied by `1.25`.

In [ ]:
expected_scale_b3 = (1 - p_b3) * 0.0 + p_b3 * scale_keep_b3 # expectation of b/p.
print("expected branch scale:", round(expected_scale_b3, 3)) # inspect the average multiplier.
assert round(expected_scale_b3, 3) == 1.000 # verify unbiased scaling.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["drop scale", "keep scale", "mean"], [0.0, scale_keep_b3, expected_scale_b3], color=["gray", "teal", "purple"])
plt.title("Basic 3: inverted scaling")
plt.ylabel("multiplier on F(x)")
plt.show()

▶ What you'll see: the mean multiplier is 1 even though individual passes are either 0 or 1.25.

👀 Takeaway: inverted scaling makes stochastic-depth training match the ordinary residual network in expectation.

### Basic 4 — Sample many survival masks

**Goal.** Simulate Bernoulli path survival, because stochastic depth is probabilistic regularization applied per example or minibatch. We build it in 2 steps.

In [ ]:
rng_b4 = np.random.default_rng(4) # create reproducible local randomness.
p_b4 = 0.75 # choose a survival probability.
masks_b4 = (rng_b4.random(1000) < p_b4).astype(float) # sample many branch masks.
print("empirical survival:", round(float(masks_b4.mean()), 3)) # inspect sampled survival rate.
assert abs(float(masks_b4.mean()) - p_b4) < 0.04 # verify sampling is close to expectation.

▶ What you'll see: the sampled survival fraction is close to 0.75.

In [ ]:
plt.figure(figsize=(4, 3))
plt.hist(masks_b4, bins=[-0.1, 0.1, 0.9, 1.1], color="darkcyan", edgecolor="black")
plt.title("Basic 4: Bernoulli survival masks")
plt.xlabel("mask value")
plt.ylabel("count")
plt.show()

▶ What you'll see: most masks are 1 and the rest are 0.

👀 Takeaway: stochastic depth turns one deterministic deep model into many sampled shallower paths during training.

### Basic 5 — Compute active depth

**Goal.** Count surviving branches in a network, because the sum of Bernoulli masks is the effective residual depth for one pass. We build it in 2 steps.

In [ ]:
p_layers_b5 = np.array([1.0, 0.9, 0.8, 0.7]) # survival probabilities from shallow to deep blocks.
rng_b5 = np.random.default_rng(5) # create deterministic random draws.
mask_b5 = (rng_b5.random(len(p_layers_b5)) < p_layers_b5).astype(int) # sample one path through the network.
print("layer masks:", mask_b5) # inspect which residual branches survived.
print("active depth:", int(mask_b5.sum())) # count surviving residual branches.

▶ What you'll see: one sampled sub-network with some number of active residual branches.

In [ ]:
expected_depth_b5 = float(p_layers_b5.sum()) # expected active branches across many passes.
print("expected active depth:", round(expected_depth_b5, 3)) # inspect the capacity budget.
plt.figure(figsize=(4, 3))
plt.bar(np.arange(1, 5), p_layers_b5, color="seagreen")
plt.title("Basic 5: survival schedule")
plt.xlabel("block")
plt.ylabel("p_survive")
plt.ylim(0, 1.05)
plt.show()

▶ What you'll see: later layers have lower survival probability, so expected active depth is less than full depth.

👀 Takeaway: survival probabilities directly control average network depth and regularization strength.

### Basic 6 — Measure a matrix's largest singular value

**Goal.** Compute `σmax(W)`, because spectral normalization needs the largest possible layer stretch. We build it in 2 steps.

In [ ]:
W_b6 = np.array([[3.0, 1.0], [0.0, 2.0]]) # define a small layer matrix.
s_b6 = np.linalg.svd(W_b6, compute_uv=False) # compute singular values exactly for the tiny matrix.
print("singular values:", np.round(s_b6, 3)) # inspect all stretches.
assert round(float(s_b6[0]), 3) == 3.257 # verify the top stretch.

▶ What you'll see: the largest singular value is about 3.257.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["σ1", "σ2"], s_b6, color="orange")
plt.title("Basic 6: singular values")
plt.ylabel("stretch factor")
plt.show()

▶ What you'll see: `σ1` is the worst-case gain that spectral normalization will cap.

👀 Takeaway: the spectral norm is the largest singular value, not an average or elementwise weight size.

### Basic 7 — Rescale a layer to unit spectral norm

**Goal.** Divide `W` by `σmax(W)`, because spectral normalization caps the layer's worst-case gain at 1. We build it in 2 steps.

In [ ]:
W_b7 = np.array([[3.0, 1.0], [0.0, 2.0]]) # define the same layer matrix.
W_bar_b7, sigma_b7 = spectral_normalize(W_b7) # apply exact spectral normalization.
print("sigma before:", round(sigma_b7, 3)) # inspect original top gain.
print("W_bar:\n", np.round(W_bar_b7, 3)) # inspect rescaled weights.

▶ What you'll see: every weight is divided by the same top singular value.

In [ ]:
top_after_b7 = np.linalg.svd(W_bar_b7, compute_uv=False)[0] # compute top gain after normalization.
print("top singular value after:", round(float(top_after_b7), 3)) # inspect capped gain.
assert round(float(top_after_b7), 3) == 1.000 # verify unit spectral norm.
plt.figure(figsize=(4, 3))
plt.bar(["before", "after"], [sigma_b7, top_after_b7], color=["crimson", "teal"])
plt.title("Basic 7: spectral norm before/after")
plt.ylabel("σmax")
plt.show()

▶ What you'll see: the top gain falls from about 3.257 to 1.

👀 Takeaway: spectral normalization preserves direction structure while enforcing a maximum layer gain.

### Basic 8 — Visualize stretch on the unit circle

**Goal.** Map unit vectors through `W` and `W_bar`, because singular values are easiest to see as circle-to-ellipse geometry. We build it in 2 steps.

In [ ]:
W_b8 = np.array([[3.0, 1.0], [0.0, 2.0]]) # define the raw layer.
W_bar_b8, sigma_b8 = spectral_normalize(W_b8) # normalize the layer.
theta_b8 = np.linspace(0, 2 * np.pi, 120) # angles around the unit circle.
circle_b8 = np.vstack([np.cos(theta_b8), np.sin(theta_b8)]) # unit vectors as columns.
print("raw sigma:", round(sigma_b8, 3)) # inspect the normalization factor.

▶ What you'll see: the raw layer has a top stretch above 3.

In [ ]:
raw_b8 = W_b8 @ circle_b8 # image of the circle under the raw layer.
norm_b8 = W_bar_b8 @ circle_b8 # image of the circle under the normalized layer.
plt.figure(figsize=(4.5, 4))
plt.plot(circle_b8[0], circle_b8[1], color="gray", label="input circle")
plt.plot(raw_b8[0], raw_b8[1], color="crimson", label="raw W")
plt.plot(norm_b8[0], norm_b8[1], color="teal", label="normalized W")
plt.axis("equal"); plt.legend(); plt.title("Basic 8: circle stretch")
plt.show()

▶ What you'll see: the raw ellipse is much larger; the normalized ellipse fits within a unit-scale bound.

👀 Takeaway: spectral normalization is geometric control over the worst stretched direction.

### Basic 9 — Run power iteration

**Goal.** Estimate the top singular value without full SVD, because real layers need cheap per-update normalization. We build it in 3 steps.

In [ ]:
W_b9 = np.array([[3.0, 1.0], [0.0, 2.0]]) # define the layer matrix.
est_b9, trace_b9 = power_sigma(W_b9, steps=6) # estimate σmax by power iteration.
exact_b9 = np.linalg.svd(W_b9, compute_uv=False)[0] # compute exact value for checking.
print("estimates:", np.round(trace_b9, 4)) # inspect convergence.

▶ What you'll see: each estimate moves toward the exact top singular value.

In [ ]:
print("final estimate:", round(est_b9, 4), "exact:", round(float(exact_b9), 4)) # compare estimate and exact SVD.
assert abs(est_b9 - exact_b9) < 0.02 # verify the cheap estimate is accurate on this tiny case.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(trace_b9, marker="o", color="purple")
plt.axhline(exact_b9, color="black", linestyle="--")
plt.title("Basic 9: power-iteration convergence")
plt.xlabel("iteration")
plt.ylabel("σ estimate")
plt.show()

▶ What you'll see: the curve approaches the dashed exact singular value.

👀 Takeaway: power iteration finds the dominant stretch using only repeated matrix-vector products.

### Basic 10 — Combine path survival and spectral rescale

**Goal.** Apply stochastic depth to a spectrally normalized branch, because the two methods constrain different failure modes: path co-adaptation and layer gain. We build it in 3 steps.

In [ ]:
x_b10 = np.array([1.5, -0.5]) # define the shortcut input.
W_b10 = np.array([[3.0, 1.0], [0.0, 2.0]]) # define a raw branch matrix.
W_bar_b10, sigma_b10 = spectral_normalize(W_b10) # cap the branch's linear gain.
F_b10 = relu(W_bar_b10 @ x_b10) # compute a normalized residual branch.
print("sigma raw:", round(sigma_b10, 3), "F after normalized W:", np.round(F_b10, 3)) # inspect both constraints.

▶ What you'll see: the branch is computed with a weight matrix whose worst-case stretch is capped.

In [ ]:
p_b10 = 0.8 # survival probability.
y_drop_b10 = stochastic_depth_output(x_b10, F_b10, 0.0, p_b10) # dropped residual branch.
y_keep_b10 = stochastic_depth_output(x_b10, F_b10, 1.0, p_b10) # surviving branch with inverted scaling.
print("drop:", np.round(y_drop_b10, 3)) # inspect shortcut-only output.
print("keep:", np.round(y_keep_b10, 3)) # inspect scaled-branch output.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["drop y0", "keep y0"], [y_drop_b10[0], y_keep_b10[0]], color=["gray", "teal"])
plt.title("Basic 10: normalized branch with stochastic depth")
plt.ylabel("output coordinate 0")
plt.show()

▶ What you'll see: stochastic depth toggles the branch while spectral normalization has already limited its gain.

👀 Takeaway: stochastic depth regularizes which paths train; spectral normalization regulates how strongly a surviving path can amplify.

## 🟡 Easy

### Easy 1 — Compare scaled and unscaled stochastic depth

**Goal.** Show why `1/p` scaling matters, because unscaled stochastic depth lowers the average residual contribution during training. We build it in 3 steps.

In [ ]:
x_e1 = np.array([1.5, -0.5]) # define a shortcut signal.
F_e1 = np.array([3.1, 0.0]) # define the residual correction.
p_e1 = 0.8 # define branch survival probability.
print("plain residual:", np.round(x_e1 + F_e1, 3)) # inspect the target test-time scale.

▶ What you'll see: the ordinary residual output is the scale we want to match in expectation.

In [ ]:
unscaled_mean_e1 = x_e1 + p_e1 * F_e1 # expectation if using y=x+bF(x).
scaled_mean_e1 = x_e1 + p_e1 * (F_e1 / p_e1) # expectation if using y=x+(b/p)F(x).
print("unscaled expectation:", np.round(unscaled_mean_e1, 3)) # inspect biased-down branch scale.
print("scaled expectation:", np.round(scaled_mean_e1, 3)) # inspect unbiased branch scale.
assert np.allclose(scaled_mean_e1, x_e1 + F_e1) # verify expectation preservation.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["plain", "unscaled mean", "scaled mean"], [(x_e1 + F_e1)[0], unscaled_mean_e1[0], scaled_mean_e1[0]], color=["black", "crimson", "teal"])
plt.title("Easy 1: expected output scale")
plt.ylabel("coordinate 0")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: unscaled dropping lowers the average; inverted scaling matches the plain residual output.

👀 Takeaway: inverted stochastic-depth scaling prevents a train-test residual-scale mismatch.

### Easy 2 — Sweep survival probability

**Goal.** Vary survival probability and measure active depth, because `p` is a regularization knob. We build it in 3 steps.

In [ ]:
p_grid_e2 = np.array([1.0, 0.9, 0.7, 0.5, 0.3]) # possible uniform survival probabilities.
L_e2 = 12 # residual blocks in the toy network.
expected_depth_e2 = L_e2 * p_grid_e2 # expected active residual branches.
print("expected depths:", np.round(expected_depth_e2, 2)) # inspect the capacity effect.

▶ What you'll see: lowering `p` shortens the average training path.

In [ ]:
rng_e2 = np.random.default_rng(22) # create reproducible sampling.
emp_depth_e2 = [] # store sampled active depths.
for p_e2 in p_grid_e2:
    masks_e2 = rng_e2.random((2000, L_e2)) < p_e2 # sample many networks at this p.
    emp_depth_e2.append(float(masks_e2.sum(axis=1).mean())) # average active depth.
print("empirical depths:", np.round(emp_depth_e2, 2)) # inspect simulated depths.
assert np.max(np.abs(np.array(emp_depth_e2) - expected_depth_e2)) < 0.25 # verify sampling matches expectation.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(p_grid_e2, expected_depth_e2, marker="o", label="expected")
plt.plot(p_grid_e2, emp_depth_e2, marker="s", label="empirical")
plt.title("Easy 2: survival probability controls depth")
plt.xlabel("survival probability p")
plt.ylabel("active branches")
plt.legend()
plt.show()

▶ What you'll see: expected and sampled active depths overlap closely.

👀 Takeaway: survival probability controls the average depth and therefore the strength of path regularization.

### Easy 3 — Bound a layer's gain

**Goal.** Compare input-output norm ratios before and after spectral normalization, because the spectral norm is the worst-case gain. We build it in 3 steps.

In [ ]:
W_e3 = np.array([[3.0, 1.0], [0.0, 2.0]]) # raw weight matrix.
W_bar_e3, sigma_e3 = spectral_normalize(W_e3) # normalized weight matrix.
rng_e3 = np.random.default_rng(33) # reproducible random test directions.
X_e3 = rng_e3.normal(size=(400, 2)) # random input vectors.
print("sigma before:", round(sigma_e3, 3)) # inspect the raw worst-case gain.

▶ What you'll see: the raw layer can stretch by more than 3× in the worst direction.

In [ ]:
gain_raw_e3 = np.linalg.norm(X_e3 @ W_e3.T, axis=1) / np.linalg.norm(X_e3, axis=1) # sampled raw gain ratios.
gain_norm_e3 = np.linalg.norm(X_e3 @ W_bar_e3.T, axis=1) / np.linalg.norm(X_e3, axis=1) # sampled normalized gain ratios.
print("max sampled raw gain:", round(float(gain_raw_e3.max()), 3)) # inspect sampled raw stretch.
print("max sampled normalized gain:", round(float(gain_norm_e3.max()), 3)) # inspect sampled capped stretch.
assert gain_norm_e3.max() <= 1.001 # verify sampled normalized gain stays at or below 1.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(gain_raw_e3, alpha=0.6, label="raw", color="crimson")
plt.hist(gain_norm_e3, alpha=0.6, label="spectral norm", color="teal")
plt.title("Easy 3: sampled gain ratios")
plt.xlabel("||Wx|| / ||x||")
plt.legend()
plt.show()

▶ What you'll see: normalized gains are compressed below 1, while raw gains can be much larger.

👀 Takeaway: spectral normalization converts a loose layer into one with a controlled Lipschitz gain.

### Easy 4 — Normalize to a chosen coefficient

**Goal.** Rescale a matrix to spectral norm `c` instead of 1, because some architectures want a specific gain budget. We build it in 3 steps.

In [ ]:
W_e4 = np.array([[2.0, -1.0], [1.0, 2.0]]) # define a matrix with equal singular values.
c_e4 = 0.9 # target spectral norm coefficient.
sigma_e4 = np.linalg.svd(W_e4, compute_uv=False)[0] # compute original top singular value.
print("original sigma:", round(float(sigma_e4), 3)) # inspect raw gain.

▶ What you'll see: the raw top gain is larger than the desired coefficient.

In [ ]:
W_c_e4 = c_e4 * W_e4 / sigma_e4 # normalize to target spectral norm c.
sigma_c_e4 = np.linalg.svd(W_c_e4, compute_uv=False)[0] # compute new top singular value.
print("target c:", c_e4, "new sigma:", round(float(sigma_c_e4), 3)) # inspect target match.
assert round(float(sigma_c_e4), 3) == 0.900 # verify chosen gain coefficient.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["raw", "target-normalized"], [sigma_e4, sigma_c_e4], color=["orange", "teal"])
plt.axhline(c_e4, color="black", linestyle="--")
plt.title("Easy 4: target spectral coefficient")
plt.ylabel("σmax")
plt.show()

▶ What you'll see: the normalized bar lands exactly on the target line.

👀 Takeaway: spectral normalization can enforce any chosen gain ceiling, not only unit gain.

### Easy 5 — Track a one-step spectral update

**Goal.** Apply a gradient step and then renormalize, because spectral normalization is enforced repeatedly during training. We build it in 3 steps.

In [ ]:
W_e5 = np.array([[3.0, 1.0], [0.0, 2.0]]) # current weight matrix.
g_e5 = np.array([[0.5, -0.2], [0.1, 0.4]]) # toy gradient.
eta_e5 = 0.08 # learning rate from the lesson's scalar-update style.
W_step_e5 = W_e5 - eta_e5 * g_e5 # unconstrained gradient step.
print("raw sigma after step:", round(float(np.linalg.svd(W_step_e5, compute_uv=False)[0]), 3)) # inspect post-update gain.

▶ What you'll see: a normal optimizer step changes both the weights and the spectral norm.

In [ ]:
W_sn_e5, sigma_step_e5 = spectral_normalize(W_step_e5) # reapply spectral normalization after the update.
print("sigma before renorm:", round(sigma_step_e5, 3)) # inspect the value used for division.
print("sigma after renorm:", round(float(np.linalg.svd(W_sn_e5, compute_uv=False)[0]), 3)) # inspect restored bound.
assert round(float(np.linalg.svd(W_sn_e5, compute_uv=False)[0]), 3) == 1.000 # verify constraint.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["after SGD", "after SN"], [sigma_step_e5, np.linalg.svd(W_sn_e5, compute_uv=False)[0]], color=["crimson", "teal"])
plt.title("Easy 5: update then spectral rescale")
plt.ylabel("σmax")
plt.show()

▶ What you'll see: renormalization restores the spectral gain cap after the gradient step.

👀 Takeaway: spectral normalization is a repeated projection-like rescale that keeps training inside a gain budget.

## 🔴 Advanced

### Advanced 1 — Compare gradient path survival across depth

**Goal.** Simulate active paths over many blocks, because stochastic depth gives gradients shorter identity routes on dropped branches. We build it in 4 steps.

In [ ]:
L_a1 = 20 # number of residual blocks.
p_a1 = np.linspace(1.0, 0.5, L_a1) # survival schedule from shallow to deep.
rng_a1 = np.random.default_rng(101) # reproducible path sampling.
masks_a1 = rng_a1.random((4000, L_a1)) < p_a1 # sample many training passes.
print("expected active blocks:", round(float(p_a1.sum()), 3)) # inspect expected depth.

▶ What you'll see: the scheduled network trains at a smaller average active depth than 20.

In [ ]:
active_a1 = masks_a1.sum(axis=1) # active residual branches per sampled pass.
print("mean active:", round(float(active_a1.mean()), 3), "std:", round(float(active_a1.std()), 3)) # inspect the depth distribution.
assert abs(float(active_a1.mean()) - float(p_a1.sum())) < 0.15 # verify sampled mean.

In [ ]:
full_depth_fraction_a1 = float(np.mean(active_a1 == L_a1)) # probability every branch survives.
short_fraction_a1 = float(np.mean(active_a1 <= 14)) # probability of much shorter path.
print("all branches survive fraction:", round(full_depth_fraction_a1, 3)) # inspect rare full-depth pass.
print("depth <= 14 fraction:", round(short_fraction_a1, 3)) # inspect regularized shallow passes.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(active_a1, bins=np.arange(active_a1.min(), active_a1.max() + 2) - 0.5, color="mediumpurple", edgecolor="black")
plt.title("Advanced 1: distribution of active residual depth")
plt.xlabel("active residual branches")
plt.ylabel("training passes")
plt.show()

▶ What you'll see: training samples a distribution of network depths rather than one fixed 20-block path.

👀 Takeaway: stochastic depth regularizes deep networks by training an ensemble of effective depths with shared weights.

### Advanced 2 — Check Lipschitz gain through multiple layers

**Goal.** Compare products of spectral norms, because a composition's worst-case gain is bounded by the product of layer gains. We build it in 4 steps.

In [ ]:
Ws_a2 = [np.array([[3.0, 1.0], [0.0, 2.0]]), np.array([[1.0, -2.0], [2.0, 1.0]]), np.array([[2.0, 0.5], [-0.5, 1.0]])] # three raw layers.
sigmas_a2 = np.array([np.linalg.svd(W_a2, compute_uv=False)[0] for W_a2 in Ws_a2]) # per-layer top gains.
print("raw layer sigmas:", np.round(sigmas_a2, 3)) # inspect gain budget per layer.
print("product bound:", round(float(np.prod(sigmas_a2)), 3)) # inspect composition bound.

▶ What you'll see: multiplying several gains can create a large worst-case amplification bound.

In [ ]:
Ws_sn_a2 = [W_a2 / np.linalg.svd(W_a2, compute_uv=False)[0] for W_a2 in Ws_a2] # spectrally normalize every layer.
sigmas_sn_a2 = np.array([np.linalg.svd(W_a2, compute_uv=False)[0] for W_a2 in Ws_sn_a2]) # verify normalized gains.
print("normalized layer sigmas:", np.round(sigmas_sn_a2, 3)) # inspect capped layer gains.
assert np.allclose(np.round(sigmas_sn_a2, 3), np.ones(3)) # verify all top gains are 1.

In [ ]:
M_raw_a2 = Ws_a2[2] @ Ws_a2[1] @ Ws_a2[0] # raw composed linear map.
M_sn_a2 = Ws_sn_a2[2] @ Ws_sn_a2[1] @ Ws_sn_a2[0] # normalized composed linear map.
gain_raw_a2 = np.linalg.svd(M_raw_a2, compute_uv=False)[0] # actual raw composition top gain.
gain_sn_a2 = np.linalg.svd(M_sn_a2, compute_uv=False)[0] # actual normalized composition top gain.
print("actual raw composition gain:", round(float(gain_raw_a2), 3)) # inspect true composed gain.
print("actual normalized composition gain:", round(float(gain_sn_a2), 3)) # inspect capped composed gain.
assert gain_sn_a2 <= 1.001 # verify composition does not exceed the product bound of 1.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["raw", "SN each layer"], [gain_raw_a2, gain_sn_a2], color=["crimson", "teal"])
plt.title("Advanced 2: composed layer gain")
plt.ylabel("σmax of composed map")
plt.show()

▶ What you'll see: normalizing each layer sharply reduces the composed worst-case gain.

👀 Takeaway: spectral normalization is powerful in depth because layer gain bounds multiply through compositions.

### Advanced 3 — Estimate σmax during a changing update

**Goal.** Track exact and power-iteration spectral norms across updates, because practical spectral normalization uses estimates while weights move. We build it in 4 steps.

In [ ]:
W_a3 = np.array([[3.0, 1.0], [0.0, 2.0]]) # starting layer.
grad_a3 = np.array([[0.25, -0.4], [0.1, 0.2]]) # fixed toy gradient direction.
eta_a3 = 0.08 # learning rate.
exact_trace_a3 = [] # store exact singular values.
power_trace_a3 = [] # store power-iteration estimates.
print("tracking 8 updates") # inspect experiment size.

▶ What you'll see: the next cells compare exact SVD with cheap estimates after each update.

In [ ]:
for step_a3 in range(8): # simulate a few optimizer steps.
    W_a3 = W_a3 - eta_a3 * grad_a3 # update weights.
    exact_a3 = float(np.linalg.svd(W_a3, compute_uv=False)[0]) # exact top singular value.
    est_a3, _ = power_sigma(W_a3, steps=6) # estimated top singular value.
    exact_trace_a3.append(exact_a3) # store exact value.
    power_trace_a3.append(est_a3) # store estimated value.
print("exact:", np.round(exact_trace_a3, 4)) # inspect exact spectral norms.
print("power:", np.round(power_trace_a3, 4)) # inspect estimates.

In [ ]:
max_err_a3 = float(np.max(np.abs(np.array(exact_trace_a3) - np.array(power_trace_a3)))) # maximum estimation error.
print("max estimation error:", round(max_err_a3, 4)) # inspect estimate quality.
assert max_err_a3 < 0.03 # verify power iteration tracks this small changing matrix.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(exact_trace_a3, marker="o", label="exact SVD", color="black")
plt.plot(power_trace_a3, marker="s", label="power iteration", color="darkorange")
plt.title("Advanced 3: σmax tracking during updates")
plt.xlabel("update")
plt.ylabel("σmax")
plt.legend()
plt.show()

▶ What you'll see: the estimated curve closely follows the exact curve as weights change.

👀 Takeaway: power iteration is accurate enough for teaching-scale matrices and cheap enough for large neural layers.

### Advanced 4 — Study stochastic-depth variance

**Goal.** Measure output variance under different survival probabilities, because inverted scaling preserves the mean but changes noise level. We build it in 4 steps.

In [ ]:
x_a4 = np.array([1.5, -0.5]) # shortcut signal.
F_a4 = np.array([3.1, 0.0]) # residual correction.
p_grid_a4 = np.array([0.9, 0.7, 0.5, 0.3]) # survival probabilities to compare.
rng_a4 = np.random.default_rng(404) # reproducible samples.
print("p grid:", p_grid_a4) # inspect the stochastic-depth strengths.

▶ What you'll see: smaller `p` means more aggressive path dropping.

In [ ]:
means_a4 = [] # store average coordinate 0.
vars_a4 = [] # store variance of coordinate 0.
for p_a4 in p_grid_a4:
    b_a4 = (rng_a4.random(4000) < p_a4).astype(float) # sample branch survival.
    y0_a4 = x_a4[0] + (b_a4 / p_a4) * F_a4[0] # coordinate 0 with inverted scaling.
    means_a4.append(float(y0_a4.mean())) # store empirical mean.
    vars_a4.append(float(y0_a4.var())) # store empirical variance.
print("means:", np.round(means_a4, 3)) # inspect mean preservation.
print("variances:", np.round(vars_a4, 3)) # inspect noise growth.

In [ ]:
plain_y0_a4 = float(x_a4[0] + F_a4[0]) # deterministic residual output coordinate.
print("plain y0:", plain_y0_a4) # inspect target mean.
assert np.max(np.abs(np.array(means_a4) - plain_y0_a4)) < 0.12 # verify means are close to deterministic output.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(p_grid_a4, vars_a4, marker="o", color="crimson")
plt.gca().invert_xaxis()
plt.title("Advanced 4: stochastic-depth variance")
plt.xlabel("survival probability p (smaller = stronger)")
plt.ylabel("output variance for coordinate 0")
plt.show()

▶ What you'll see: the mean stays near the ordinary residual output, but variance rises as survival probability falls.

👀 Takeaway: stochastic depth regularizes by injecting path noise; lower survival probabilities increase that noise even with unbiased scaling.

### Advanced 5 — Put both constraints in a tiny training loop

**Goal.** Train one toy residual branch with and without constraints, because stochastic depth and spectral normalization change optimization behavior through masks and gain caps. We build it in 5 steps.

In [ ]:
X_a5 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 1.0]]) # tiny batch of two-dimensional inputs.
Y_a5 = np.array([[1.2, 0.0], [0.0, 0.8], [1.1, 0.9], [-0.8, 0.7]]) # target residual-block outputs.
W_free_a5 = np.array([[1.4, 0.6], [-0.2, 1.2]]) # unconstrained branch weights.
W_sn_a5 = W_free_a5.copy() # constrained branch weights start identically.
print("batch size:", X_a5.shape[0]) # inspect tiny training set.

▶ What you'll see: we train a residual mapping `x + ReLU(Wx)` against four targets.

In [ ]:
eta_a5 = 0.05 # learning rate.
p_a5 = 0.8 # stochastic-depth survival probability.
loss_free_a5 = [] # store unconstrained losses.
loss_sn_a5 = [] # store constrained losses.
print("eta:", eta_a5, "p_survive:", p_a5) # inspect hyperparameters.

In [ ]:
for step_a5 in range(80): # run simple full-batch gradient descent by finite differences.
    pred_free_a5 = X_a5 + relu(X_a5 @ W_free_a5.T) # unconstrained residual predictions.
    err_free_a5 = pred_free_a5 - Y_a5 # residual errors.
    active_free_a5 = (X_a5 @ W_free_a5.T) > 0 # ReLU derivative mask.
    grad_free_a5 = ((err_free_a5 * active_free_a5).T @ X_a5) / X_a5.shape[0] # branch-weight gradient.
    W_free_a5 = W_free_a5 - eta_a5 * grad_free_a5 # unconstrained update.
    pred_sn_a5 = X_a5 + relu(X_a5 @ W_sn_a5.T) # constrained model uses expected branch for loss tracking.
    err_sn_a5 = pred_sn_a5 - Y_a5 # constrained residual errors.
    active_sn_a5 = (X_a5 @ W_sn_a5.T) > 0 # ReLU derivative mask.
    survive_scale_a5 = 1.0 / p_a5 if (step_a5 % 5 != 0) else 0.0 # deterministic keep/drop pattern with mean near stochastic behavior.
    grad_sn_a5 = survive_scale_a5 * ((err_sn_a5 * active_sn_a5).T @ X_a5) / X_a5.shape[0] # path-scaled branch gradient.
    W_sn_a5 = W_sn_a5 - eta_a5 * grad_sn_a5 # gradient step.
    W_sn_a5, _ = spectral_normalize(W_sn_a5) # enforce spectral gain cap after each update.
    loss_free_a5.append(float(np.mean(err_free_a5 ** 2))) # record free loss.
    loss_sn_a5.append(float(np.mean(err_sn_a5 ** 2))) # record constrained loss.
print("final free loss:", round(loss_free_a5[-1], 4), "final constrained loss:", round(loss_sn_a5[-1], 4)) # inspect endpoints.

In [ ]:
sigma_free_a5 = float(np.linalg.svd(W_free_a5, compute_uv=False)[0]) # top gain of free branch.
sigma_sn_a5 = float(np.linalg.svd(W_sn_a5, compute_uv=False)[0]) # top gain of constrained branch.
print("free sigma:", round(sigma_free_a5, 3), "constrained sigma:", round(sigma_sn_a5, 3)) # inspect gain caps.
assert round(sigma_sn_a5, 3) == 1.000 # verify spectral normalization remains active.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(loss_free_a5, label="free branch", color="gray")
plt.plot(loss_sn_a5, label="stochastic-depth + SN", color="teal")
plt.title("Advanced 5: constrained residual training")
plt.xlabel("step")
plt.ylabel("mean squared error")
plt.legend()
plt.show()

▶ What you'll see: both models learn, but the constrained branch keeps its spectral gain fixed at 1 while stochastic-depth-style updates add path noise.

👀 Takeaway: stochastic depth and spectral normalization are compatible constraints: one randomizes residual path usage, the other caps surviving-path amplification.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Stochastic depth regularizes paths; spectral normalization caps layer gain by controlling the largest singular value.

Deep residual paths can over-amplify signals; path sampling and singular-value caps constrain that amplification. Save a copy to Drive to edit.

In [ ]:

import math
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def pad_to_64(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 64:
        return X.copy()
    if X.shape[1] > 64:
        return X[:, :64].copy()
    out = np.zeros((X.shape[0], 64), dtype=float)
    out[:, :X.shape[1]] = X
    return out


def split_scale(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te




def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def train_small_mlp_loss(X, y, random_state=0, max_iter=90):
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    clf = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", alpha=0.001, max_iter=max_iter, random_state=random_state)
    clf.fit(x_tr, y_tr)
    proba = clf.predict_proba(x_te)
    loss = log_loss(y_te, proba, labels=clf.classes_)
    acc = accuracy_score(y_te, clf.predict(x_te))
    return float(loss), float(acc), clf


def train_small_mlp_acc(X, y, random_state=0, max_iter=90):
    loss, acc, clf = train_small_mlp_loss(X, y, random_state=random_state, max_iter=max_iter)
    return acc, loss, clf


def preview_ladder(rungs):
    for rung, (name, X, y) in enumerate(rungs, 1):
        classes = np.unique(y)
        print(f"D{rung}: {name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")


def plot_metric_table(rows, metric_name):
    print(f"{'rung':<4} {'dataset':<36} {metric_name:>10}")
    for row in rows:
        print(f"{row['rung']:<4} {row['name']:<36} {row['metric']:10.4f}")


def plot_summary(rows, metric_name, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["rung"] for row in rows]
    values = [row["metric"] for row in rows]
    axes[0].bar(names, values, color="steelblue")
    axes[0].set_title("Metric by ladder rung")
    axes[0].set_ylabel(ylabel)
    axes[1].plot(names, values, marker="o")
    axes[1].set_title("D1→D5 trend")
    axes[1].set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def show_artifacts(rungs, title):
    fig, axes = plt.subplots(1, len(rungs), figsize=(13, 2.6))
    for ax, (name, X, y) in zip(axes, rungs):
        if X.shape[1] == 64:
            ax.imshow(X[0].reshape(8, 8), cmap="gray")
            ax.set_title(name.split()[0])
        else:
            ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="tab10")
            ax.set_title(name.split()[0])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## The concept, built once

The lesson combines $y=x+bF(x)$ with $\bar W=W/\sigma_{max}(W)$. For the scalar pass, $1.6\cdot1.5+0.2\cdot(-0.5)+0.8=3.1$ and $2.0-0.08\cdot1.5=1.88$.

In [ ]:

def residual_path(x, Fx, b):
    return x + b * Fx


def spectral_normalize(W):
    sigma = np.linalg.svd(W, compute_uv=False)[0]
    return W / sigma, float(sigma)

x = np.array([1.0, -1.0])
Fx = np.array([0.25, 0.5])
y_live = residual_path(x, Fx, 1)
y_drop = residual_path(x, Fx, 0)
W = np.array([[3.0, 0.0], [0.0, 4.0]])
Wn, sigma = spectral_normalize(W)
updated = 2.0 - 0.08 * 1.5
print("survived path", y_live)
print("dropped path", y_drop)
print("sigma", sigma)
print("normalized sigma", np.linalg.svd(Wn, compute_uv=False)[0])
assert np.allclose(y_live, np.array([1.25, -0.5]))
assert np.allclose(y_drop, x)
assert np.isclose(sigma, 4.0)
assert np.isclose(updated, 1.88)


The reusable sweep trains a small classifier and applies the requested stability knobs while holding the ladder fixed.

In [ ]:

def regularizer_sweep(X, y, stochastic_depth=True, spectral_norm=True):
    X64 = pad_to_64(X)
    x_tr, x_te, y_tr, y_te = split_scale(X64, y)
    rng = np.random.default_rng(23)
    if stochastic_depth:
        mask = rng.binomial(1, 0.8, size=x_tr.shape)
        x_aug = x_tr + mask * np.tanh(x_tr)
    else:
        x_aug = x_tr + np.tanh(x_tr)
    if spectral_norm:
        scale = np.linalg.svd(x_aug[: min(80, len(x_aug))], compute_uv=False)[0]
        x_aug = x_aug / max(scale, 1.0)
        x_eval = (x_te + np.tanh(x_te)) / max(scale, 1.0)
    else:
        x_eval = x_te + np.tanh(x_te)
    clf = LogisticRegression(max_iter=1200)
    clf.fit(x_aug, y_tr)
    preds = clf.predict(x_eval)
    return float(accuracy_score(y_te, preds))


## The dataset ladder

The same code is run from a four-point XOR problem through real 8×8 digit images and a noisy shifted D5 variant.

In [ ]:
rungs = clf_digits_ladder()
preview_ladder(rungs)
show_artifacts(rungs, 'Ladder preview')

## Run the SAME method across D1–D5

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(rungs, 1):
    acc = regularizer_sweep(X, y, stochastic_depth=True, spectral_norm=True)
    rows.append({"rung": f"D{idx}", "name": name, "metric": acc})
plot_metric_table(rows, "accuracy")


## Results visualization

In [ ]:
show_artifacts(rungs, 'Regularizer artifacts')
plot_summary(rows, 'accuracy', 'held-out accuracy')

## Pitfall on D5: ignoring scale

In [ ]:

name, X, y = rungs[-1]
X64 = pad_to_64(X) * 80.0
unstable = regularizer_sweep(X64, y, stochastic_depth=False, spectral_norm=False)
stable = regularizer_sweep(X64, y, stochastic_depth=True, spectral_norm=True)
print("without constraints", unstable)
print("with stochastic depth + spectral norm", stable)
assert stable >= unstable - 0.05


## Evaluate it + Practice

- Metric: held-out accuracy; compare it with a majority-class or plain logistic baseline.
- Sanity check: D1 should be hand-checkable before trusting D5.
- Ablation: turn off the key idea and the reported metric should get worse or the pitfall should reappear.
- Failure signals: unstable losses, collapsed routing, inflated gradient error, or a D5 result that ignores label noise.

Practice 1: change one hyperparameter and rerun the D1 assert before touching D5.

Practice 2: add a baseline row to the ladder table and explain the largest gap.

Practice 3: make the D5 pitfall more severe, then show the same fix still helps.